# 04.3 TensorRT-LLM: Compilation, Build & Engine Comparison

Hands-on exploration of TensorRT-LLM's compilation workflow, build command generation,
engine performance comparison matrix, and a decision function for TRT-LLM vs vLLM selection.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import json
import time
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from content.utils.benchmark import BenchmarkResult
from content.utils.latency import LatencyTracker

## TensorRT-LLM Compilation Workflow

The TRT-LLM pipeline: **HuggingFace checkpoint → Convert to TRT-LLM format → Build TRT engine → Deploy**

Each stage applies optimizations: weight quantization during conversion, kernel fusion and
memory planning during engine build, and batching strategy at deploy time.

In [ ]:
@dataclass
class TRTLLMConfig:
    """Configuration for a TensorRT-LLM engine build."""
    model_name: str
    tp_size: int = 1
    pp_size: int = 1
    dtype: str = "float16"
    quantization: Optional[str] = None  # None, 'int8_weight_only', 'int4_awq', 'fp8'
    max_batch_size: int = 64
    max_input_len: int = 2048
    max_output_len: int = 512
    max_beam_width: int = 1
    use_inflight_batching: bool = True
    use_paged_kv_cache: bool = True
    kv_cache_free_gpu_fraction: float = 0.9
    enable_chunked_context: bool = True
    builder_opt_level: int = 5  # 0-5, higher = slower build, faster inference


class CompilationWorkflow:
    """Models the TRT-LLM compilation stages and estimates."""

    # Approximate build times (minutes) per billion parameters at opt_level=5
    BUILD_TIME_PER_B = {"float16": 8, "int8_weight_only": 12, "int4_awq": 15, "fp8": 10}

    STAGES = [
        "download_checkpoint",
        "convert_to_trtllm_format",
        "build_trt_engine",
        "validate_engine",
    ]

    def __init__(self, config: TRTLLMConfig, model_size_b: float):
        self.config = config
        self.model_size_b = model_size_b

    def estimate_build_time_minutes(self) -> float:
        base = self.BUILD_TIME_PER_B.get(self.config.quantization or self.config.dtype, 8)
        return base * self.model_size_b * (self.config.builder_opt_level / 5)

    def estimate_engine_size_gb(self) -> float:
        bytes_per_param = {"float16": 2, "int8_weight_only": 1, "int4_awq": 0.5, "fp8": 1}
        bpp = bytes_per_param.get(self.config.quantization or self.config.dtype, 2)
        return self.model_size_b * bpp * 1.1  # 10% overhead for metadata/buffers

    def print_workflow(self):
        print(f"{'='*60}")
        print(f"TRT-LLM Build Workflow: {self.config.model_name}")
        print(f"{'='*60}")
        print(f"Model size: {self.model_size_b}B params | TP={self.config.tp_size} PP={self.config.pp_size}")
        print(f"Quantization: {self.config.quantization or 'none (fp16)'}")
        print(f"Opt level: {self.config.builder_opt_level}/5")
        print(f"\nStages:")
        for i, stage in enumerate(self.STAGES, 1):
            print(f"  {i}. {stage}")
        print(f"\nEstimated build time: {self.estimate_build_time_minutes():.0f} min")
        print(f"Estimated engine size: {self.estimate_engine_size_gb():.1f} GB")
        print(f"Engine shards: {self.config.tp_size * self.config.pp_size}")


# Demo: Llama-70B with INT4 AWQ on 4 GPUs
cfg = TRTLLMConfig(
    model_name="meta-llama/Llama-3-70B",
    tp_size=4, quantization="int4_awq",
    max_batch_size=128, max_input_len=4096
)
workflow = CompilationWorkflow(cfg, model_size_b=70)
workflow.print_workflow()

In [ ]:
class BuildCommandGenerator:
    """Generates trtllm-build CLI commands from config."""

    def generate_convert_cmd(self, config: TRTLLMConfig, output_dir: str = "/tmp/trtllm_ckpt") -> str:
        cmd = [
            "python convert_checkpoint.py",
            f"--model_dir /models/{config.model_name}",
            f"--output_dir {output_dir}",
            f"--dtype {config.dtype}",
            f"--tp_size {config.tp_size}",
            f"--pp_size {config.pp_size}",
        ]
        if config.quantization == "int8_weight_only":
            cmd.append("--use_weight_only --weight_only_precision int8")
        elif config.quantization == "int4_awq":
            cmd.append("--use_weight_only --weight_only_precision int4_awq")
        elif config.quantization == "fp8":
            cmd.append("--enable_fp8 --fp8_kv_cache")
        return " \\\n    ".join(cmd)

    def generate_build_cmd(self, config: TRTLLMConfig,
                           ckpt_dir: str = "/tmp/trtllm_ckpt",
                           engine_dir: str = "/tmp/trtllm_engine") -> str:
        cmd = [
            "trtllm-build",
            f"--checkpoint_dir {ckpt_dir}",
            f"--output_dir {engine_dir}",
            f"--max_batch_size {config.max_batch_size}",
            f"--max_input_len {config.max_input_len}",
            f"--max_seq_len {config.max_input_len + config.max_output_len}",
            f"--max_beam_width {config.max_beam_width}",
            f"--builder_opt {config.builder_opt_level}",
        ]
        if config.use_paged_kv_cache:
            cmd.append("--use_paged_context_fmha enable")
        if config.enable_chunked_context:
            cmd.append("--enable_chunked_context")
        return " \\\n    ".join(cmd)

    def generate_full_pipeline(self, config: TRTLLMConfig) -> str:
        lines = ["#!/bin/bash", f"# TRT-LLM build pipeline for {config.model_name}", ""]
        lines.append("# Step 1: Convert checkpoint")
        lines.append(self.generate_convert_cmd(config))
        lines.append("\n# Step 2: Build engine")
        lines.append(self.generate_build_cmd(config))
        return "\n".join(lines)


gen = BuildCommandGenerator()
print(gen.generate_full_pipeline(cfg))

## Engine Comparison Matrix

Compare TRT-LLM engine configurations across key metrics: throughput, latency,
memory footprint, and build complexity.

In [ ]:
@dataclass
class EngineProfile:
    """Simulated performance profile for a TRT-LLM engine config."""
    name: str
    quantization: str
    tp_size: int
    throughput_tok_s: float  # tokens/sec at max batch
    ttft_ms: float          # time to first token
    itl_ms: float           # inter-token latency
    memory_gb: float        # GPU memory per device
    build_time_min: float
    accuracy_drop_pct: float  # vs FP16 baseline


def build_comparison_matrix(model_size_b: float) -> pd.DataFrame:
    """Generate comparison matrix for common engine configurations."""
    # Simulated profiles based on published TRT-LLM benchmarks (scaled by model size)
    scale = model_size_b / 7  # normalize to 7B baseline
    profiles = [
        EngineProfile("FP16-TP1", "none", 1, 2800/scale, 45*scale, 12*scale, 14*scale/1, 8*scale, 0.0),
        EngineProfile("FP16-TP2", "none", 2, 4900/scale, 28*scale, 7.5*scale, 14*scale/2, 10*scale, 0.0),
        EngineProfile("INT8-TP1", "int8_wo", 1, 4100/scale, 35*scale, 9*scale, 7.5*scale/1, 12*scale, 0.3),
        EngineProfile("INT4-AWQ-TP1", "int4_awq", 1, 5200/scale, 32*scale, 8*scale, 4.2*scale/1, 15*scale, 0.8),
        EngineProfile("FP8-TP1", "fp8", 1, 4600/scale, 33*scale, 8.5*scale, 7.5*scale/1, 10*scale, 0.1),
        EngineProfile("INT4-AWQ-TP4", "int4_awq", 4, 18000/scale, 18*scale, 4*scale, 4.2*scale/4, 20*scale, 0.8),
    ]
    rows = [{
        "Config": p.name, "Quant": p.quantization, "TP": p.tp_size,
        "Throughput (tok/s)": f"{p.throughput_tok_s:.0f}",
        "TTFT (ms)": f"{p.ttft_ms:.1f}", "ITL (ms)": f"{p.itl_ms:.1f}",
        "Mem/GPU (GB)": f"{p.memory_gb:.1f}",
        "Build (min)": f"{p.build_time_min:.0f}",
        "Acc Drop %": f"{p.accuracy_drop_pct:.1f}"
    } for p in profiles]
    return pd.DataFrame(rows)


print("Engine Comparison Matrix — Llama-3-7B equivalent")
print("=" * 80)
df = build_comparison_matrix(7.0)
print(df.to_string(index=False))
print("\n--- Scaled to 70B ---")
df70 = build_comparison_matrix(70.0)
print(df70.to_string(index=False))

In [ ]:
@dataclass
class WorkloadRequirements:
    """Describes the deployment workload to guide engine selection."""
    model_size_b: float
    target_throughput_tok_s: float
    max_latency_ttft_ms: float
    max_latency_itl_ms: float
    available_gpus: int
    gpu_memory_gb: float = 80.0  # per GPU
    accuracy_tolerance_pct: float = 1.0
    batch_variability: str = "high"  # low, medium, high
    request_pattern: str = "mixed"   # chat, batch, mixed
    need_fast_iteration: bool = False  # frequent model updates?


def decide_trt_vs_vllm(req: WorkloadRequirements) -> Dict:
    """Decision function: recommends TRT-LLM or vLLM based on workload."""
    scores = {"trt_llm": 0, "vllm": 0}
    reasons = {"trt_llm": [], "vllm": []}

    # Latency-critical → TRT-LLM (kernel fusion, custom CUDA)
    if req.max_latency_itl_ms < 15:
        scores["trt_llm"] += 3
        reasons["trt_llm"].append("Tight ITL budget favors TRT-LLM kernel fusion")

    # High throughput at scale → TRT-LLM
    if req.target_throughput_tok_s > 5000 * (req.available_gpus / 4):
        scores["trt_llm"] += 2
        reasons["trt_llm"].append("High throughput target benefits from TRT optimizations")

    # Fast iteration needed → vLLM (no build step)
    if req.need_fast_iteration:
        scores["vllm"] += 3
        reasons["vllm"].append("Frequent model updates — vLLM has no compilation step")

    # Variable batch sizes → vLLM (continuous batching more flexible)
    if req.batch_variability == "high":
        scores["vllm"] += 2
        reasons["vllm"].append("High batch variability suits vLLM's continuous batching")

    # Memory constrained → quantized TRT-LLM
    mem_needed_fp16 = req.model_size_b * 2  # GB
    mem_available = req.available_gpus * req.gpu_memory_gb
    if mem_needed_fp16 > mem_available * 0.6:
        scores["trt_llm"] += 2
        reasons["trt_llm"].append("Memory-constrained — TRT-LLM quantization more mature")

    # Accuracy sensitive → vLLM (simpler to validate, no compilation artifacts)
    if req.accuracy_tolerance_pct < 0.5:
        scores["vllm"] += 2
        reasons["vllm"].append("Very tight accuracy tolerance — fewer quantization surprises")

    # Chat workload → both good, slight edge to TRT-LLM for TTFT
    if req.request_pattern == "chat":
        scores["trt_llm"] += 1
        reasons["trt_llm"].append("Chat pattern benefits from TRT-LLM's optimized TTFT")

    # Batch/offline → vLLM simpler to set up
    if req.request_pattern == "batch":
        scores["vllm"] += 1
        reasons["vllm"].append("Batch workloads don't need TRT-LLM's latency optimizations")

    winner = "trt_llm" if scores["trt_llm"] > scores["vllm"] else "vllm"
    if scores["trt_llm"] == scores["vllm"]:
        winner = "vllm"  # default to simpler option on tie
        reasons["vllm"].append("Tie-breaker: vLLM is simpler to operate")

    return {
        "recommendation": winner,
        "scores": scores,
        "reasons": reasons,
        "confidence": abs(scores["trt_llm"] - scores["vllm"]) / max(sum(scores.values()), 1)
    }


# Scenario 1: Latency-critical chat service
req1 = WorkloadRequirements(
    model_size_b=70, target_throughput_tok_s=8000,
    max_latency_ttft_ms=100, max_latency_itl_ms=10,
    available_gpus=4, batch_variability="medium", request_pattern="chat"
)
result1 = decide_trt_vs_vllm(req1)
print("Scenario 1: Latency-critical chat (70B, 4 GPUs)")
print(f"  Recommendation: {result1['recommendation'].upper()}")
print(f"  Scores: TRT-LLM={result1['scores']['trt_llm']} vs vLLM={result1['scores']['vllm']}")
print(f"  Confidence: {result1['confidence']:.0%}")
for r in result1['reasons'][result1['recommendation']]:
    print(f"    • {r}")

print()

# Scenario 2: Research team iterating on fine-tuned models
req2 = WorkloadRequirements(
    model_size_b=7, target_throughput_tok_s=2000,
    max_latency_ttft_ms=500, max_latency_itl_ms=50,
    available_gpus=1, batch_variability="high",
    request_pattern="batch", need_fast_iteration=True
)
result2 = decide_trt_vs_vllm(req2)
print("Scenario 2: Research iteration (7B, 1 GPU, frequent updates)")
print(f"  Recommendation: {result2['recommendation'].upper()}")
print(f"  Scores: TRT-LLM={result2['scores']['trt_llm']} vs vLLM={result2['scores']['vllm']}")
for r in result2['reasons'][result2['recommendation']]:
    print(f"    • {r}")

## Key Takeaways

| Factor | TRT-LLM Wins | vLLM Wins |
|--------|-------------|----------|
| Latency | ✅ Kernel fusion, custom CUDA | |
| Throughput at scale | ✅ Optimized memory layout | |
| Iteration speed | | ✅ No build step |
| Operational simplicity | | ✅ pip install + go |
| Quantization maturity | ✅ FP8, INT4 AWQ native | |
| Variable workloads | | ✅ Continuous batching |
| Multi-model serving | | ✅ No per-model compilation |

**Rule of thumb**: Use TRT-LLM when you have a stable model serving latency-sensitive production traffic.
Use vLLM when you need flexibility, fast experimentation, or serve multiple models.

In [ ]:
# Quick reference: generate build commands for common configurations
gen = BuildCommandGenerator()
configs = [
    ("7B-FP16", TRTLLMConfig("Llama-3-8B", tp_size=1)),
    ("70B-INT4", TRTLLMConfig("Llama-3-70B", tp_size=4, quantization="int4_awq")),
    ("70B-FP8", TRTLLMConfig("Llama-3-70B", tp_size=4, quantization="fp8", max_batch_size=128)),
]
for name, c in configs:
    print(f"\n{'='*50}")
    print(f"Config: {name}")
    print(f"{'='*50}")
    print(gen.generate_build_cmd(c))